In [1]:
%pip install pandas pyarrow

Note: you may need to restart the kernel to use updated packages.


# Для всех датасетов в начале файла разделить их на 
- A с фактором соседства с В
- B с фактором соседства с А
- A * B
### И обрабатывать по отдельнтости каждый своей моделью

Например, у Fligts и Operational есть фактор flight
нужно сделать 3 датафрейма
1. Flights с доп бинарным признаком "Есть ли данные обо мне в Operational"
2. Operational с доп бинарным признаком "Есть ли данные обо мне в Flights"
3. Их пересечение

Потом будут рассматриваться как отдельные датасеты

Также и для Boarding и для всех

По итогу выйдут 7 датасетов исходных с соседствами
и всевозможные их пересечения (попарные, по тройкам и так далее)

выделять пересечения в отдельный датасет можно лишь если > 1000

Pipeline

- нормализовать файлы [ ]
- обработать их группы [ ]

Группировать по тройкам и так далее УЖЕ В САМОМ КОНЦЕ (ПОКА НЕ ВЫПОЛНЕНО)
В БОАРДИНГ СЕРИЯ ПАСПОРТА (ОБЕ КОЛОНКИ) НАДО НОРМАЛИЗОВАТЬ !!!!!!!!!!!!!!!!!!!!!!

In [2]:
import pandas as pd

path = "../../resources/processed/temp/"

BoardingDF = pd.read_csv(f"{path}BoardingData_parsed.csv", sep="|", engine="pyarrow")
FlightsDF = pd.read_csv(f"{path}json_parsed.csv", sep="|", engine="pyarrow")
OperationalDF = pd.read_csv(f"{path}yaml_parsed.csv", sep="|", engine="pyarrow")
SirenaDF = pd.read_csv(f"{path}sirena_parsed.csv", sep="|", engine="pyarrow")
BoardingPassDF = pd.read_csv(f"{path}cleaned_boarding_pass.csv", sep="|", engine="pyarrow")
LoyalityDF = pd.read_csv(f"{path}xml_parsed.csv", sep="|", engine="pyarrow")
TimetableDF = pd.read_csv(f"{path}Skyteam_Timetable.csv", sep="|", engine="pyarrow")

# Пересечения

In [3]:
import pandas as pd


def prepare_intersection_datasets(df1, keys1, df2, keys2, col_name):
  df1_res = df1.copy()
  df2_res = df2.copy()

  df2_keys_unique = df2[keys2].drop_duplicates()
  if keys1 != keys2:
    df2_keys_unique = df2_keys_unique.rename(columns=dict(zip(keys2, keys1)))

  df1_res = df1_res.merge(
      df2_keys_unique, on=keys1, how="left", indicator=True
  )
  df1_res[col_name] = (df1_res["_merge"] == "both").astype('int8')
  df1_res = df1_res.drop(columns=["_merge"])

  df1_keys_unique = df1[keys1].drop_duplicates()
  if keys1 != keys2:
    df1_keys_unique = df1_keys_unique.rename(columns=dict(zip(keys1, keys2)))

  df2_res = df2_res.merge(
      df1_keys_unique, on=keys2, how="left", indicator=True
  )
  df2_res[col_name] = (df2_res["_merge"] == "both").astype('int8')
  df2_res = df2_res.drop(columns=["_merge"])

  if keys1 == keys2:
    df12 = pd.merge(df1, df2, on=keys1, how="inner")
  else:
    df2_temp = df2.rename(columns=dict(zip(keys2, keys1)))
    df12 = pd.merge(df1, df2_temp, on=keys1, how="inner")

  return df1_res, df2_res, df12

In [4]:
BoardingDF['TicketNumber'] = BoardingDF['TicketNumber'].astype(str)
BoardingPassDF['Номер_эл_билета'] = BoardingPassDF['Номер_эл_билета'].astype(str)


flights, boarding, fligts_boarding = prepare_intersection_datasets(
    FlightsDF, ['Flight', 'Date'], BoardingDF, ['FlightNumber', 'FlightDate'], "boardFlightMatch"
)

boarding, operational, boarding_operational = prepare_intersection_datasets(
    boarding,  ['FlightNumber', 'FlightDate'], OperationalDF, ['Flight', 'Date'], "OperatFlightMatch"
)

flights, operational, flights_operational = prepare_intersection_datasets(
    flights, ['Flight', 'Date'], operational, ['Flight', 'Date'], "FlightOperationalMatch"
)

# flights_operational, boarding_operational, flights_boarding_operational = prepare_intersection_datasets(
#     flights_operational, ['Flight', 'Date'], boarding_operational, ['FlightNumber', 'FlightDate'], "OperationalFlightsBoardingMatch"
# )

boardingPass, boarding, pass_boarding = prepare_intersection_datasets(
    BoardingPassDF, ['Номер_эл_билета'], boarding, ['TicketNumber'], "PassBoardingMatch"
)

sirena, boarding, sirena_boarding = prepare_intersection_datasets(
    SirenaDF, ['FlightCode', 'DepartDate'], boarding, ['FlightNumber', 'FlightDate'], "SirenaBoardingMatch"
)

sirena, flights, sirena_flights = prepare_intersection_datasets(
    sirena, ['FlightCode', 'DepartDate'], flights, ['Flight', 'Date'], "SirenaFlightsMatch"
)

loyalty, operational, loyalty_operational = prepare_intersection_datasets(
    LoyalityDF, ['Code', 'Date'], operational, ['Flight', 'Date'], "LoyaltyOperationalMatch"
)

loyalty, flights, loyalty_flights = prepare_intersection_datasets(
    loyalty, ['Code', 'Date'], flights, ['Flight', 'Date'], "LoyaltyFlightsMatch"
)


# timetable, flights, timetable_flights = prepare_intersection_datasets(
#     TimetableDF, ['FLIGHT'], flights, ['Flight'], "TimetableFlightsMatch"
# )

In [5]:
BoardingDF = boarding
FlightsDF = flights
OperationalDF = operational
SirenaDF = sirena
BoardingPassDF = boardingPass
LoyalityDF = loyalty
# TimetableDF = timetable

# Нормализация BoardingDF

In [6]:
display(BoardingDF)

,PassengerFirstName,PassengerSecondName,PassengerLastName,PassengerSex,PassengerBirthDate,PassengerDocument,BookingCode,TicketNumber,Baggage,FlightDate,FlightTime,FlightNumber,CodeShare,Destination,boardFlightMatch,OperatFlightMatch,PassBoardingMatch,SirenaBoardingMatch
0,SAVELII,VIKTOROVICH,RUSANOV,Male,03/10/1983,2879 096860,FRNINO,6625956945991971,Transit,2017-03-22,06:05:00,SU1369,Own,Moscow,0,1,0,0
1,LEV,MARKOVICH,ISAEV,Male,12/13/1975,1788 173211,Not presented,1643715499224676,Registered,2017-03-18,22:10:00,SU1180,Own,Volgograd,1,1,1,0
2,NIKOLAI,I.,NIKOLSKII,Male,12/26/1990,4396 926588,VWNYGF,6247422701565929,Transit,2017-03-18,22:10:00,SU1180,Own,Volgograd,1,1,1,0
3,ANATOLII,PETROVICH,SHILOV,Male,05/24/1997,2595 919752,WQFFUE,Not presented,Registered,2017-03-18,22:10:00,SU1180,Own,Volgograd,1,1,0,0
4,MIROSLAVA,VIACHESLAVOVNA,SEMENOVA,Female,01/31/1976,6775 516990,Not presented,Not presented,Registered,2017-03-12,11:45:00,SU6284,Own,Moscow,1,1,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
155142,RAMIL,S.,DOROFEEV,Male,02/14/1994,3256 077151,SNRBXU,8653871661040452,Delayed,2017-05-18,06:30:00,SU1158,Own,Rostov,1,1,0,0
155143,RUSLAN,KONSTANTINOVICH,CHISTIAKOV,Male,02/14/1975,3497 458395,Not presented,0223584175158947,NaN,2017-05-18,06:30:00,SU1158,Own,Rostov,1,1,0,0
155144,SOFIA,VALEREVNA,KALMYKOVA,Female,12/17/1972,6618 352713,Not presented,Not presented,Registered,2017-10-02,07:30:00,SU5614,Own,Petropavlovsk,1,1,0,0
155145,VIACHESLAV,F.,KONONOV,Male,05/20/2000,4567 242088,Not presented,0975693658645383,Registered,2017-09-16,07:00:00,SU1640,Own,Omsk,1,1,0,0


In [7]:
BoardingDF["PassengerFirstName"] = BoardingDF['PassengerFirstName'].map(
    BoardingDF['PassengerFirstName'].value_counts(normalize=True)
)
BoardingDF["PassengerSecondName"] = BoardingDF['PassengerSecondName'].map(
    BoardingDF['PassengerSecondName'].value_counts(normalize=True)
)
BoardingDF["PassengerLastName"] = BoardingDF['PassengerLastName'].map(
    BoardingDF['PassengerLastName'].value_counts(normalize=True)
)

BoardingDF["PassengerSex"] = BoardingDF["PassengerSex"].map({'Female': 0, 'Male': 1})

BoardingDF['PassengerBirthDate'] = pd.to_datetime(BoardingDF['PassengerBirthDate'], format='%m/%d/%Y')
BoardingDF['FlightDate'] = pd.to_datetime(BoardingDF['FlightDate'], format='%Y-%m-%d')
BoardingDF['PassengerAge'] = ((BoardingDF['FlightDate'] - BoardingDF['PassengerBirthDate']).dt.days // 365.25).astype(int)

passport_series = BoardingDF['PassengerDocument'].astype(str).str.slice(0, 4)
BoardingDF['PassportSeriesLeft'] = passport_series.str.slice(0, 2)
BoardingDF['PassportSeriesRight'] = passport_series.str.slice(2, 4)

BoardingDF["BookingCode"] = (BoardingDF['BookingCode'] != 'Not presented').astype('int8')

BoardingDF['Baggage'] = BoardingDF['Baggage'].fillna('No')

BoardingDF['BaggageTransit'] = (BoardingDF['Baggage'] == 'Transit').astype('int8')
BoardingDF['BaggageDelayed'] = (BoardingDF['Baggage'] == 'Delayed').astype('int8')
BoardingDF['BaggageRegistered'] = (BoardingDF['Baggage'] == 'Registered').astype('int8')

BoardingDF['Flight_DayOfWeek'] = BoardingDF['FlightDate'].dt.dayofweek
BoardingDF['Flight_Month'] = BoardingDF['FlightDate'].dt.month

ft = BoardingDF['FlightTime'].astype(str)

BoardingDF['FlightTime'] = (
    ft.str.slice(0, 2).astype(int) * 60 +
    ft.str.slice(3, 5).astype(int)
)

BoardingDF["FlightNumber"] = BoardingDF['FlightNumber'].map(
    BoardingDF['FlightNumber'].value_counts(normalize=True)
)

BoardingDF["TicketNumber"] = (BoardingDF['TicketNumber'] != 'Not presented').astype(int)

BoardingDF["Destination"] = BoardingDF['Destination'].map(
    BoardingDF['Destination'].value_counts(normalize=True)
)

BoardingDF["CodeShare"] = (BoardingDF['CodeShare'] != 'Operated').astype(int)

BoardingDF = BoardingDF.drop(columns=['PassengerBirthDate', 'FlightDate', 'PassengerDocument', 'Baggage', 'FlightDate'])

In [8]:
display(BoardingDF)

,PassengerFirstName,PassengerSecondName,PassengerLastName,PassengerSex,BookingCode,TicketNumber,FlightTime,FlightNumber,CodeShare,Destination,...,PassBoardingMatch,SirenaBoardingMatch,PassengerAge,PassportSeriesLeft,PassportSeriesRight,BaggageTransit,BaggageDelayed,BaggageRegistered,Flight_DayOfWeek,Flight_Month
0,0.004654,0.002662,0.000786,1,1,1,365,0.004428,1,0.567346,...,0,0,34,28,79,1,0,0,2,3
1,0.004673,0.002424,0.000857,1,0,1,1330,0.001476,1,0.008914,...,1,0,41,17,88,0,0,1,5,3
2,0.004834,0.044899,0.001225,1,1,1,1330,0.001476,1,0.008914,...,1,0,26,43,96,1,0,0,5,3
3,0.005343,0.002494,0.000851,1,1,0,1330,0.001476,1,0.008914,...,0,0,19,25,95,0,0,1,5,3
4,0.004944,0.002598,0.000838,0,0,0,705,0.002849,1,0.567346,...,0,0,41,67,75,0,0,1,6,3
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
155142,0.005027,0.045389,0.001141,1,1,1,390,0.001424,1,0.011273,...,0,0,23,32,56,0,1,0,3,5
155143,0.004918,0.002462,0.000709,1,0,1,390,0.001424,1,0.011273,...,0,0,42,34,97,0,0,0,3,5
155144,0.005292,0.002475,0.000928,0,0,0,450,0.001959,1,0.010848,...,0,0,44,66,18,0,0,1,0,10
155145,0.004589,0.009926,0.001044,1,0,1,420,0.001334,1,0.004460,...,0,0,17,45,67,0,0,1,5,9


# Нормализация FlightsDF

In [9]:
display(FlightsDF)

,Date,Codeshare,Flight,DepCity,DepAirport,DepCountry,ArrCity,ArrAirport,ArrCountry,boardFlightMatch,FlightOperationalMatch,SirenaFlightsMatch,LoyaltyFlightsMatch
0,2017-03-04,False,DL3377,Boston MA,BOS,United States of,Pittsburgh PA,PIT,United States of,0,1,0,0
1,2017-07-21,False,DL3630,Green Bay WI,GRB,United States of,Detroit MI,DTW,United States of,0,1,0,0
2,2017-03-22,False,DL5361,Pittsburgh PA,PIT,United States of,New York NY,LGA,United States of,0,1,0,0
3,2017-01-04,True,SU3702,Moscow,SVO,Russian Federation,Warsaw,WAW,Poland,0,1,0,0
4,2017-08-08,False,DL3412,Fayetteville AR,XNA,United States of,Atlanta GA,ATL,United States of,0,1,0,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...
1308982,2017-12-15,False,DL3345,Washington,IAD,United States of,New York NY,JFK,United States of,0,0,0,0
1308983,2017-08-12,False,SU2014,Moscow,SVO,Russian Federation,Prague,PRG,Czech Republic,0,0,0,0
1308984,2017-12-29,False,DL2463,Atlanta GA,ATL,United States of,New York NY,LGA,United States of,0,0,0,0
1308985,2017-06-02,False,SU1463,Novosibirsk,OVB,Russian Federation,Moscow,SVO,Russian Federation,1,1,0,0


In [10]:
FlightsDF['Date'] = pd.to_datetime(FlightsDF['Date'])
FlightsDF['Flight_DayOfWeek'] = FlightsDF['Date'].dt.dayofweek
FlightsDF['Flight_Month'] = FlightsDF['Date'].dt.month

FlightsDF['Codeshare'] = FlightsDF['Codeshare'].astype(int)

FlightsDF["Flight"] = FlightsDF['Flight'].map(
    FlightsDF['Flight'].value_counts(normalize=True)
)

FlightsDF['Airport'] = FlightsDF['DepAirport'] + "_" + FlightsDF['ArrAirport']
FlightsDF["Airport"] = FlightsDF['Airport'].map(
    FlightsDF['Airport'].value_counts(normalize=True)
)

FlightsDF['City'] = FlightsDF['DepCity'] + "_" + FlightsDF['ArrCity']
FlightsDF["City"] = FlightsDF['City'].map(
    FlightsDF['City'].value_counts(normalize=True)
)

FlightsDF['Country'] = FlightsDF['DepCountry'] + "_" + FlightsDF['ArrCountry']
FlightsDF["Country"] = FlightsDF['Country'].map(
    FlightsDF['Country'].value_counts(normalize=True)
)

FlightsDF = FlightsDF.drop(columns=['Date', 'DepAirport', 'ArrAirport', 'DepCity', 'ArrCity', 'DepCountry', 'ArrCountry'])

In [11]:
display(FlightsDF)

,Codeshare,Flight,boardFlightMatch,FlightOperationalMatch,SirenaFlightsMatch,LoyaltyFlightsMatch,Flight_DayOfWeek,Flight_Month,Airport,City,Country
0,0,0.000083,0,1,0,0,5,3,0.000129,0.000129,0.129043
1,0,0.000058,0,1,0,0,4,7,0.000167,0.000167,0.129043
2,0,0.000032,0,1,0,0,2,3,0.000334,0.000474,0.129043
3,1,0.000186,0,1,0,0,2,1,0.000905,0.000905,0.000912
4,0,0.000062,0,1,0,1,1,8,0.000202,0.000202,0.129043
...,...,...,...,...,...,...,...,...,...,...,...
1308982,0,0.000080,0,0,0,0,4,12,0.000316,0.000723,0.129043
1308983,0,0.000168,0,0,0,0,5,8,0.003615,0.003615,0.005781
1308984,0,0.000024,0,0,0,0,4,12,0.000395,0.000827,0.129043
1308985,0,0.000342,1,1,0,0,4,6,0.001390,0.001390,0.119576


# Нормализация OperationalDF

In [12]:
display(OperationalDF)

,Date,Flight,From,To,Status,FF_Airline,FF_Number,Class,Fare,OperatFlightMatch,FlightOperationalMatch,LoyaltyOperationalMatch
0,2017-01-01,AF1145,SVO,CDG,LANDED,FB,520518073,Y,YRSTFN,0,1,0
1,2017-01-01,AF1844,SVO,CDG,LANDED,FB,18427809,Y,YGRPZR,0,1,1
2,2017-01-01,AF4855,SVO,CDG,LANDED,KE,993927664,Y,YRSTJQ,0,1,0
3,2017-01-01,AF4901,SVO,CDG,LANDED,SU,789402704,Y,YGRPAT,0,0,0
4,2017-01-01,AZ545,SVO,FCO,LANDED,KE,352734424,Y,YFLXKO,0,0,1
...,...,...,...,...,...,...,...,...,...,...,...,...
1687048,2018-01-01,VN8067,SGN,VCS,LANDED,DT,182808803,Y,YGRPMT,0,1,0
1687049,2018-01-01,VN8067,SGN,VCS,LANDED,FB,88616357,Y,YFLXDJ,0,1,0
1687050,2018-01-01,VN813,SGN,REP,LANDED,DT,574004748,Y,YGRPAU,0,1,0
1687051,2018-01-01,VN8204,HAN,DIN,LANDED,KE,425304412,Y,YSTNDC,0,0,0


In [13]:
OperationalDF['Date'] = pd.to_datetime(OperationalDF['Date'])
OperationalDF['Flight_DayOfWeek'] = OperationalDF['Date'].dt.dayofweek
OperationalDF['Flight_Month'] = OperationalDF['Date'].dt.month

OperationalDF["Flight"] = OperationalDF['Flight'].map(
    OperationalDF['Flight'].value_counts(normalize=True)
)

OperationalDF['Airport'] = OperationalDF['From'] + "_" + OperationalDF['To']
OperationalDF["Airport"] = OperationalDF['Airport'].map(
    OperationalDF['Airport'].value_counts(normalize=True)
)

OperationalDF['ClassY'] = (OperationalDF['Class'] == 'Y').astype('int8')
OperationalDF['ClassJ'] = (OperationalDF['Class'] == 'J').astype('int8')
OperationalDF['ClassP'] = (OperationalDF['Class'] == 'P').astype('int8')

OperationalDF['FF_Airline_SU'] = (OperationalDF['FF_Airline'] == 'SU').astype('int8')
OperationalDF['FF_Airline_KE'] = (OperationalDF['FF_Airline'] == 'KE').astype('int8')
OperationalDF['FF_Airline_DT'] = (OperationalDF['FF_Airline'] == 'DT').astype('int8')

OperationalDF["FF_Number"] = OperationalDF['FF_Number'].map(
    OperationalDF['FF_Number'].value_counts(normalize=True)
)

OperationalDF["Fare"] = OperationalDF['Fare'].map(
    OperationalDF['Fare'].value_counts(normalize=True)
)

OperationalDF = OperationalDF.drop(columns=['Date', 'Status', 'From', 'To', 'Class', "FF_Airline"])

In [14]:
display(OperationalDF)

,Flight,FF_Number,Fare,OperatFlightMatch,FlightOperationalMatch,LoyaltyOperationalMatch,Flight_DayOfWeek,Flight_Month,Airport,ClassY,ClassJ,ClassP,FF_Airline_SU,FF_Airline_KE,FF_Airline_DT
0,0.000167,0.000015,0.000309,0,1,0,6,1,0.005871,1,0,0,0,0,0
1,0.000226,0.000005,0.000318,0,1,1,6,1,0.005871,1,0,0,0,0,0
2,0.000182,0.000011,0.000302,0,1,0,6,1,0.005871,1,0,0,0,1,0
3,0.000191,0.000008,0.000307,0,0,0,6,1,0.005871,1,0,0,1,0,0
4,0.000181,0.000002,0.000311,0,0,1,6,1,0.003165,1,0,0,0,1,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1687048,0.000027,0.000014,0.000346,0,1,0,0,1,0.000207,1,0,0,0,0,1
1687049,0.000027,0.000007,0.000301,0,1,0,0,1,0.000207,1,0,0,0,0,0
1687050,0.000024,0.000016,0.000295,0,1,0,0,1,0.000115,1,0,0,0,0,1
1687051,0.000037,0.000011,0.000315,0,0,0,0,1,0.000072,1,0,0,0,1,0


# Нормализация BoardingPassDF

In [15]:
display(BoardingPassDF)

,Очередь_регистрации,Класс_обслуживания,Номер_рейса,Город_вылета,Город_прилета,Код_аэропорта_вылета,Код_аэропорта_прилета,Дата_вылета,Время_вылета,PNR_бронь,Номер_эл_билета,Бонусная_карта,Фамилия,Имя,Отчество,PassBoardingMatch
0,32,Y,SU5436,VLADIVOSTOK,SEOUL,VVO,REA,2017-01-01,11:25:00,DYELAO,7100246988860481,NaN,ZHDANOVA,LIDIIA,NaN,0
1,50,Y,SU5436,VLADIVOSTOK,SEOUL,VVO,REA,2017-01-01,11:25:00,JWARRD,7608904175905333,NaN,SOKOLOV,SVIATOSLAV,NaN,0
2,32,Y,SU5436,VLADIVOSTOK,SEOUL,VVO,REA,2017-01-01,11:25:00,SETJJH,527257587351371,NaN,SUKHANOV,SVIATOSLAV,NaN,0
3,77,Y,SU5436,VLADIVOSTOK,SEOUL,VVO,REA,2017-01-01,11:25:00,MZZNJI,3696048365585673,NaN,ZELENIN,DENIS,NaN,0
4,87,Y,SU5436,VLADIVOSTOK,SEOUL,VVO,REA,2017-01-01,11:25:00,NJNNPC,7470610229847684,NaN,OSTROVSKAIA,IRINA,NaN,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
615766,54,Y,AZ7240,MILAN,JEDDAH,MXP,JED,2018-01-01,11:00:00,BAJUBQ,8673981301035301,NaN,KORNILOV,VLADIMIR,T,0
615767,9,Y,DL1161,DAYTON OH,ATLANTA GA,DAY,ATL,2018-01-01,14:17:00,SXQFLS,6057453743742357,NaN,SHESTAKOVA,NELLI,A,0
615768,47,P,SU1502,MOSCOW,TYUMEN,SVO,TJM,2018-01-01,22:30:00,VMNBHI,4424048314480896,NaN,KOLPAKOVA,MARIIA,NaN,1
615769,49,Y,GA312,JAKARTA,SURABAYA,CGK,SUB,2018-01-01,11:05:00,LTVLUP,3825472788916248,NaN,IGNATOVA,ALINA,NaN,0


In [16]:
# print(set(BoardingPassDF['Фамилия']))
print(BoardingPassDF['Номер_рейса'].value_counts().head(10))

Номер_рейса
SU4606    1355
SU4600     897
SU4602     778
SU1465     741
SU1479     727
SU1459     716
SU1491     709
SU1383     698
SU1521     670
SU1523     641
Name: count, dtype: int64


In [17]:
BoardingPassDF['ClassY'] = (BoardingPassDF['Класс_обслуживания'] == 'Y').astype('int8')
BoardingPassDF['ClassJ'] = (BoardingPassDF['Класс_обслуживания'] == 'J').astype('int8')
BoardingPassDF['ClassP'] = (BoardingPassDF['Класс_обслуживания'] == 'P').astype('int8')

BoardingPassDF['Queue'] = pd.to_numeric(
    BoardingPassDF['Очередь_регистрации']
)

BoardingPassDF['Дата_вылета'] = pd.to_datetime(BoardingPassDF['Дата_вылета'])
BoardingPassDF['Flight_DayOfWeek'] = BoardingPassDF['Дата_вылета'].dt.dayofweek
BoardingPassDF['Flight_Month'] = BoardingPassDF['Дата_вылета'].dt.month

BoardingPassDF['City'] = BoardingPassDF['Город_вылета'] + "_" + BoardingPassDF['Город_прилета']
BoardingPassDF["City"] = BoardingPassDF['City'].map(
    BoardingPassDF['City'].value_counts(normalize=True)
)

BoardingPassDF["PassengerFirstName"] = BoardingPassDF['Имя'].map(
    BoardingPassDF['Имя'].value_counts(normalize=True)
)
BoardingPassDF["PassengerSecondName"] = BoardingPassDF['Фамилия'].map(
    BoardingPassDF['Фамилия'].value_counts(normalize=True)
)
BoardingPassDF["PassengerLastName"] = BoardingPassDF['Отчество'].notna().astype('int8')

BoardingPassDF['Airport'] = BoardingPassDF['Код_аэропорта_вылета'] + "_" + BoardingPassDF['Код_аэропорта_прилета']
BoardingPassDF["Airport"] = BoardingPassDF['Airport'].map(
    BoardingPassDF['Airport'].value_counts(normalize=True)
)

BoardingPassDF["Flight"] = BoardingPassDF['Номер_рейса'].map(
    BoardingPassDF['Номер_рейса'].value_counts(normalize=True)
)

ft = BoardingPassDF['Время_вылета'].astype(str)

BoardingPassDF['FlightTime'] = (
    ft.str.slice(0, 2).astype(int) * 60 +
    ft.str.slice(3, 5).astype(int)
)

BoardingPassDF['BonusCard'] = BoardingPassDF['Бонусная_карта'].notna().astype('int8')

BoardingPassDF = BoardingPassDF.drop(columns=['Отчество', 'Время_вылета', 'Номер_рейса', 'Имя', 'Фамилия', 'Дата_вылета', 'Бонусная_карта', 'PNR_бронь', 'Очередь_регистрации', 'Класс_обслуживания', 'Код_аэропорта_прилета', 'Код_аэропорта_вылета', 'Город_вылета', 'Город_прилета', 'Номер_эл_билета' ])

In [18]:
display(BoardingPassDF)

,PassBoardingMatch,ClassY,ClassJ,ClassP,Queue,Flight_DayOfWeek,Flight_Month,City,PassengerFirstName,PassengerSecondName,PassengerLastName,Airport,Flight,FlightTime,BonusCard
0,0,1,0,0,32,6,1,0.002462,0.005379,0.000867,0,0.002462,0.000239,685,0
1,0,1,0,0,50,6,1,0.002462,0.005096,0.000805,0,0.002462,0.000239,685,0
2,0,1,0,0,32,6,1,0.002462,0.005096,0.000843,0,0.002462,0.000239,685,0
3,0,1,0,0,77,6,1,0.002462,0.005016,0.000919,0,0.002462,0.000239,685,0
4,0,1,0,0,87,6,1,0.002462,0.005059,0.000965,0,0.002462,0.000239,685,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
615766,0,1,0,0,54,0,1,0.000133,0.004604,0.001171,1,0.000133,0.000065,660,0
615767,0,1,0,0,9,0,1,0.000049,0.005221,0.000963,1,0.000049,0.000024,857,0
615768,1,0,0,1,47,0,1,0.000698,0.004835,0.001099,0,0.000698,0.000200,1350,0
615769,0,1,0,0,49,0,1,0.000352,0.004838,0.001111,0,0.000352,0.000015,665,0


# Нормализация SirenaDF

In [19]:
display(SirenaDF)

,PaxName,PaxBirthDate,DepartDate,DepartTime,ArrivalDate,ArrivalTime,FlightCode,ShFrom,Dest,Code,...,TravelDoc,Seat,Meal,TrvCls,Fare,Baggage,PaxAdditionalInfo,AgentInfo,SirenaBoardingMatch,SirenaFlightsMatch
0,ОЗЕРОВ ИЛЬДАР ДАНИИЛОВИЧ,1999-05-15,2017-05-30,00:05:00,2017-05-30,08:05:00,SU1306NO,SVO,OVB,ZBQSPY,...,053270 N/A,NaN,J,J,GRPGN0PC,S,FF#SU 38116280,Go2See,0,0
1,КОЛОСОВ САМИР ТАМЕРЛАНОВИЧ,None,2017-12-27,02:15:00,2017-12-27,04:40:00,SU1323NO,MMK,SVO,MHPBBX,...,645520 N/A,KSML,Y,Y,RSTUQ,9,FF#FB 284903754,NaN,0,0
2,ИГНАТОВА СНЕЖАНА КОНСТАНТИНОВНА,None,2017-09-19,06:40:00,2017-09-19,07:45:00,SU1481NO,KJA,SVO,REDLVB,...,961316 N/A,NaN,Y,Y,STNJL,F,NaN,KupiBilet,0,0
3,ЖАРОВ ПЛАТОН АЛЬБЕРТОВИЧ,1999-05-02,2017-03-18,22:10:00,2017-03-19,01:05:00,SU1180NO,SVO,VOG,NSJNGQ,...,865148 N/A,NaN,J,J,FLXLS0PC,#,FF#FB 884556993,Travelgenio,0,0
4,НИКОЛЬСКИЙ НИКОЛАЙ ИГОРЕВИЧ,1990-12-26,2017-03-18,22:10:00,2017-03-19,01:05:00,SU1180NO,SVO,VOG,VWNYGF,...,926588 N/A,NaN,Y,Y,FLXPG,NaN,FF#SU 183142068,OZON.travel,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
155752,ЕМЕЛЬЯНОВ МИРОСЛАВ СВЯТОСЛАВОВИЧ,None,2017-02-14,03:10:00,2017-02-14,04:00:00,SU1181NO,VOG,SVO,JYYXED,...,319326 N/A,STML,Y,Y,STNER,F,NaN,OneTwoTrip,0,0
155753,ТАРАСОВ АРТЕМИЙ ГЕРМАНОВИЧ,None,2017-02-14,03:10:00,2017-02-14,04:00:00,SU1181NO,VOG,SVO,XDKDZF,...,078129 N/A,NaN,Y,Y,FLXKV1PC,T,FF#DT 261604825,OZON.travel,0,0
155754,ГОРОХОВ БОРИС СВЯТОГОРОВИЧ,1970-10-28,2017-02-14,03:10:00,2017-02-14,04:00:00,SU1181NO,VOG,SVO,SOWGLC,...,760618 N/A,NaN,Y,Y,FLXBF,F,FF#FB 902862806,Aeroflot,0,0
155755,ДОРОФЕЕВ РАМИЛЬ СВЯТОСЛАВОВИЧ,1994-02-14,2017-05-18,06:30:00,2017-05-18,08:35:00,SU1158NO,SVO,ROV,SNRBXU,...,077151 N/A,NaN,Y,Y,GRPMD,#,NaN,NaN,0,0
